# Practice Lab: Data Preparation for Fine-Tuning (Lesson 9.2)

Module 9 · 8 exercises · Format conversion + quality scoring + dedup + PII + fertility

Exercises 1, 2, 3, 4, 5, 7 run locally (pure Python).


# Lesson 9.2: Data Preparation for Fine-Tuning

8 exercises: 2E + 3M + 3C

Exercises 1, 2, 3, 4, 5, 7 run **locally** (pure Python). Ex 6, 8 are design.


In [ ]:
import json, hashlib, re
import numpy as np
np.random.seed(42)


---
## Exercise 1 (Easy): Format Conversion (Alpaca -> OpenAI)


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import json

def alpaca_to_msgs(ex, sys=""):
    msgs = []
    if sys: msgs.append({"role":"system","content":sys})
    u = ex["instruction"]
    if ex.get("input","").strip(): u += f"\nInput: {ex['input']}"
    msgs.append({"role":"user","content":u})
    msgs.append({"role":"assistant","content":ex["output"]})
    return {"messages": msgs}

data = [
    {"instruction":"Refund policy?","input":"","output":"Full refund 7 days. 50% 8-30. None after 30."},
    {"instruction":"Summarize","input":"GenAI: 14 modules, 146 hours.","output":"A 146-hour GenAI course with 14 modules."},
    {"instruction":"Cost?","input":"","output":"14999 rupees with lifetime access."},
    {"instruction":"Translate to Hindi","input":"Hello how are you?","output":"Namaste aap kaise hain?"},
    {"instruction":"Prerequisites?","input":"","output":"Basic Python and high school math."},
    {"instruction":"Classify sentiment","input":"Amazing course!","output":"Positive"},
    {"instruction":"EMI?","input":"","output":"Yes EMI via Razorpay 2500/month."},
    {"instruction":"Live classes?","input":"","output":"Daily 7 PM IST with recordings in 2 hours."},
    {"instruction":"Extract entities","input":"Netsetos in Hyderabad","output":"ORG: Netsetos, LOC: Hyderabad"},
    {"instruction":"Completion rate?","input":"","output":"85% completion rate with 4.8 stars."},
]

sys_p = "You are a helpful Netsetos assistant."
converted = [alpaca_to_msgs(ex, sys_p) for ex in data]

with open("/tmp/converted.jsonl","w") as f:
    for item in converted: f.write(json.dumps(item)+"\n")

w_input = sum(1 for ex in data if ex.get("input","").strip())
print(f"Format Conversion: {len(converted)} examples")
print(f"With input: {w_input} | Without: {len(data)-w_input}")

errors = 0
for item in converted:
    m = item["messages"]
    if m[0]["role"]!="system" or m[1]["role"]!="user" or m[-1]["role"]!="assistant": errors += 1
print(f"Validation: {errors} errors")
print(f"JSONL saved. Alpaca=single-turn, Messages=multi-turn ready.")


</details>


---
## Exercise 2 (Easy): Chat Template Inspection


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
print("Chat Template Comparison:")

templates = {
    "Llama 3": {"header":"<|start_header_id|>{role}<|end_header_id|>","eot":"<|eot_id|>"},
    "Mistral": {"inst_start":"[INST]","inst_end":"[/INST]","note":"No system prompt v0.1-0.2"},
    "Qwen 2.5": {"header":"<|im_start|>{role}","eot":"<|im_end|>"},
    "Gemma 2": {"turn":"<start_of_turn>{role}","end":"<end_of_turn>","note":"No system; prepend to user"},
}

for model, t in templates.items():
    print(f"\n  [{model}]")
    for k,v in t.items(): print(f"    {k}: {v}")

print(f"\nSame messages produce DIFFERENT token sequences!")
print(f"Wrong template = silent failure (no error, bad output)")
print(f"ChatBug (2024): can bypass safety alignment")
print(f"FIX: Always use tokenizer.apply_chat_template()")


</details>


---
## Exercise 3 (Medium): Quality Scoring Pipeline


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import numpy as np
np.random.seed(42)

def score_qa(u, a):
    stops = {"what","is","the","a","how","do","for","in","to","of","and","can","you"}
    uw = set(u.lower().split()) - stops; aw = set(a.lower().split())
    acc = min(round(len(uw&aw)/max(len(uw),1)*5,1),5.0)
    w = len(a.split())
    hlp = 1.0 if w<5 else 3.0 if w<15 else 4.0 if w<40 else 5.0
    comp = min(2.0+("." in a)+any(c.isdigit() for c in a)+(w>10),5.0)
    return round((acc+hlp+comp)/3,2)

examples = [
    ("Refund policy?","Full refund 7 days. 50% 8-30 days. None after 30."),
    ("Cost?","14999"),("Prerequisites?","Basic Python and high school math. No ML needed."),
    ("Help me","ok"),("Tools?","Colab, Vertex AI, ChromaDB, LangChain, Docker."),
    ("EMI?","Yes EMI via Razorpay starting 2500 rupees per month for 6 months."),
    ("Live classes?","Daily 7 PM IST. Recordings within 2 hours."),
    ("Certificate?","yes we give certificate"),
    ("Duration?","146 hours across 14 modules. Python, ML, GenAI, deployment."),
    ("Group discount?","20% off for groups of 3 or more."),
    ("What is RAG?","It's a thing."),
    ("Fine-tuning?","Fine-tuning adapts a pre-trained model to specific task using labeled examples."),
    ("Support email?","Email us"),
    ("LoRA?","LoRA trains only 0.1-5% of params by adding rank decomposition matrices. 6-10GB VRAM."),
    ("Students?","Over 5000 enrolled with 85% completion and 4.8 stars."),
]

scored = sorted([(u,a,score_qa(u,a)) for u,a in examples], key=lambda x:-x[2])
th = 2.0
kept = [(u,a,s) for u,a,s in scored if s>=th]
dropped = [(u,a,s) for u,a,s in scored if s<th]

print("Quality Scoring:")
print(f"Kept ({len(kept)}):")
for u,a,s in kept[:4]: print(f"  [{s:.1f}] {u[:20]}... {a[:30]}...")
print(f"Dropped ({len(dropped)}):")
for u,a,s in dropped: print(f"  [{s:.1f}] {u[:20]}... {a[:30]}...")

all_s=[s for _,_,s in scored]; kpt_s=[s for _,_,s in kept]
print(f"\nMean: {np.mean(all_s):.2f} -> Kept: {np.mean(kpt_s):.2f} (+{(np.mean(kpt_s)/np.mean(all_s)-1)*100:.0f}%)")
print(f"AlpaGasus: 9K filtered > 52K unfiltered")



</details>


---
## Exercise 4 (Medium): Deduplication Pipeline


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import hashlib

class Dedup:
    def __init__(self, jt=0.6): self.jt=jt
    def _h(self,t): return hashlib.sha256(t.encode()).hexdigest()
    def _ng(self,t,n=3):
        w=t.lower().split(); return set(tuple(w[i:i+n]) for i in range(len(w)-n+1))
    def _jac(self,a,b):
        if not a or not b: return 0.0
        return len(a&b)/len(a|b)
    def run(self, examples):
        seen={}; after=[]; ed=0
        for ex in examples:
            h=self._h(ex["output"])
            if h in seen: ed+=1
            else: seen[h]=1; after.append(ex)
        kept=[]; kng=[]; nd=0
        for ex in after:
            ng=self._ng(ex["output"]); dup=False
            for p in kng:
                if self._jac(ng,p)>=self.jt: dup=True; break
            if dup: nd+=1
            else: kept.append(ex); kng.append(ng)
        return ed, nd, kept

examples = [
    {"i":"Refund?","output":"Full refund within 7 days. 50% from 8-30. None after 30 days."},
    {"i":"Cost?","output":"14999 rupees with lifetime access."},
    {"i":"Prereqs?","output":"Basic Python and high school math."},
    {"i":"Refund?","output":"Full refund within 7 days. 50% from 8-30. None after 30 days."},
    {"i":"Price?","output":"14999 rupees with lifetime access to all course materials."},
    {"i":"EMI?","output":"Yes EMI via Razorpay starting 2500 per month."},
    {"i":"Prereqs?","output":"Basic Python and high school math."},
    {"i":"Classes?","output":"Daily 7 PM IST with recordings in 2 hours."},
    {"i":"Return?","output":"Full refund within 7 days of purchase. 50% from 8 to 30 days. None after 30 days."},
    {"i":"Duration?","output":"146 hours across 14 modules."},
    {"i":"Cert?","output":"Yes completion certificate provided."},
    {"i":"EMI?","output":"Yes EMI via Razorpay starting 2500 per month."},
]

ed,nd,kept = Dedup(0.6).run(examples)
print(f"Deduplication Pipeline:")
print(f"  Original: {len(examples)} | Exact dups: {ed} | Near dups: {nd} | Kept: {len(kept)}")
print(f"  Reduction: {(1-len(kept)/len(examples))*100:.0f}%")
for ex in kept: print(f"    {ex['i']:<12} {ex['output'][:35]}...")
print(f"\nExact=SHA-256, Near=n-gram Jaccard. Production: MinHash/LSH")


</details>


---
## Exercise 5 (Medium): PII Removal Pipeline


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import re

class PII:
    PATS = {
        "EMAIL": r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b',
        "PHONE": r'\b(?:\+91[-\s]?)?[6-9]\d{9}\b',
        "AADHAAR": r'\b\d{4}\s\d{4}\s\d{4}\b',
        "PAN": r'\b[A-Z]{5}[0-9]{4}[A-Z]\b',
    }
    def remove(self, text):
        r = text; found = {}
        for typ, pat in self.PATS.items():
            ms = re.findall(pat, r)
            if ms: found[typ]=len(ms)
            for m in ms: r = r.replace(m, f"<{typ}>")
        return r, found

pii = PII()
texts = [
    "Contact help@netsetos.com for refunds.",
    "Call +91 9876543210 for enrollment.",
    "Aadhaar: 1234 5678 9012 for verification.",
    "PAN: ABCDE1234F for tax.",
    "Email admin@netsetos.com or call 9123456789.",
    "Student 1234 5678 9012 with PAN XYZAB5678C.",
    "No PII in this course description.",
    "Contact 8765432109 for group discount.",
]

print("PII Removal:")
total = 0
for t in texts:
    c, f = pii.remove(t)
    total += sum(f.values())
    if f: print(f"  {t[:40]}... -> {c[:40]}... ({f})")
    else: print(f"  [clean] {t[:45]}...")

print(f"\nTotal PII found: {total}")
print(f"Replacement preserves sentence structure")
print(f"DPDPA 2023: penalties up to 250 crore")


</details>


---
## Exercise 7 (Challenge): Tokenizer Fertility Benchmark


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
print("Tokenizer Fertility Benchmark:")

fert = {
    "English": {"Llama3":1.3,"Mistral":1.3,"Gemma2":1.3,"Sarvam1":1.4},
    "Hindi": {"Llama3":5.2,"Mistral":6.1,"Gemma2":2.2,"Sarvam1":1.6},
    "Telugu": {"Llama3":11.7,"Mistral":12.3,"Gemma2":4.8,"Sarvam1":2.1},
}

print(f"{'Lang':<8} {'Llama3':>8} {'Mistral':>8} {'Gemma2':>8} {'Sarvam1':>8}")
print("-"*42)
for lang, ms in fert.items():
    best = min(ms, key=ms.get)
    row = f"  {lang:<6}"
    for m,f in ms.items(): row += f" {f:>7.1f}{'*' if m==best else ' '}"
    print(row)

print(f"\nContext impact (4096 tokens):")
for lang in ["Hindi","Telugu"]:
    lw = int(4096/fert[lang]["Llama3"]); sw = int(4096/fert[lang]["Sarvam1"])
    print(f"  {lang}: Llama3={lw} words, Sarvam1={sw} words ({sw/lw:.1f}x)")

print(f"\nRule: Choose model by tokenizer fertility FIRST")
print(f"Telugu on Llama3 = 11.7 tok/word -> effectively 9x more expensive")


</details>


---
## Exercise 6 (Challenge): Synthetic Data Pipeline
Design/architecture. See practice-lab-9_2.html.


In [ ]:
# Exercise 6: Synthetic Data Pipeline
pass


---
## Exercise 8 (Challenge): Full Production Pipeline
Design/architecture. See practice-lab-9_2.html.


In [ ]:
# Exercise 8: Full Production Pipeline
pass
